# Data collection for the year 2010

In [1]:
import cocopp
dsl = cocopp.load("bbob/2010/*")

In [2]:
import numpy as np

dd = dsl.dictByDimFunc()     # your grouped datasets
t = 1e-8                     # choose the target precision

best_by_df = {}              # (dim, fid) -> (best_alg, best_ert)

for dim in sorted(dd.keys()): 
    for fid in sorted(dd[dim].keys()):
        rows = []
        for ds in dd[dim][fid]:                 # each ds = one algorithm
            ert = float(ds.detERT([t])[0])      # ERT in #evals at target t
            rows.append((ds.algId, ert))  
        # ignore INF (not reached) when picking best
        finite = [(a, e) for (a, e) in rows if np.isfinite(e)] 
    
        if finite:
            best_alg, best_ert = min(finite, key=lambda x: x[1]) 
        else:
            best_alg, best_ert = None, np.inf
        best_by_df[(dim, fid)] = (best_alg, best_ert) 
        print(f"dim={dim:>2}, F{fid:>2} -> {best_alg}  (ERT={best_ert:.3g} @ {t})")


dim= 2, F 1 -> AVGNEWUOA_ros  (ERT=6 @ 1e-08)
dim= 2, F 2 -> 1plus2mirser_brockhoff  (ERT=434 @ 1e-08)
dim= 2, F 3 -> AVGNEWUOA_ros  (ERT=1.45e+03 @ 1e-08)
dim= 2, F 4 -> ABC_elabd  (ERT=3.48e+03 @ 1e-08)
dim= 2, F 5 -> AVGNEWUOA_ros  (ERT=7.07 @ 1e-08)
dim= 2, F 6 -> 1plus2mirser_brockhoff  (ERT=270 @ 1e-08)
dim= 2, F 7 -> 1plus1_brockhoff  (ERT=203 @ 1e-08)
dim= 2, F 8 -> AVGNEWUOA_ros  (ERT=146 @ 1e-08)
dim= 2, F 9 -> AVGNEWUOA_ros  (ERT=175 @ 1e-08)
dim= 2, F10 -> 1plus2mirser_brockhoff  (ERT=429 @ 1e-08)
dim= 2, F11 -> 1plus2mirser_brockhoff  (ERT=428 @ 1e-08)
dim= 2, F12 -> AVGNEWUOA_ros  (ERT=289 @ 1e-08)
dim= 2, F13 -> IPOP-ACTCMA-ES_ros  (ERT=828 @ 1e-08)
dim= 2, F14 -> 1plus2mirser_brockhoff  (ERT=428 @ 1e-08)
dim= 2, F15 -> DEuniform_fialho  (ERT=1.51e+03 @ 1e-08)
dim= 2, F16 -> IPOP-ACTCMA-ES_ros  (ERT=1.48e+03 @ 1e-08)
dim= 2, F17 -> IPOP-ACTCMA-ES_ros  (ERT=1.9e+03 @ 1e-08)
dim= 2, F18 -> PM-AdapSS-DE_fialho  (ERT=2.35e+03 @ 1e-08)
dim= 2, F19 -> MOS_torre  (ERT=1.7e+03 @

In [3]:
from collections import Counter, defaultdict

In [4]:
# Build a frequency counter: how many (dim,fid) each algo wins
win_counter = Counter(
    alg for (alg, ert) in best_by_df.values()
    if alg is not None and np.isfinite(ert)
)

# If you want a plain dict:
wins_dict = dict(win_counter)

# (Optional) pretty print, most wins first
for alg, count in win_counter.most_common():
    print(f"{alg}: {count}")

IPOP-ACTCMA-ES_ros: 45
AVGNEWUOA_ros: 31
1plus2mirser_brockhoff: 19
MOS_torre: 11
IPOP-CMA-ES_ros: 9
ABC_elabd: 8
1plus1_brockhoff: 5
1komma4mirser_brockhoff: 5
DEuniform_fialho: 3
PM-AdapSS-DE_fialho: 2
DE-F-AUC_fialho: 1
NBC-CMA_preuss: 1


In [5]:
"""
Given best_by_df: {(dim, fid): (alg, ert)},
return {dim: algo_with_most_(fid)_wins_in_that_dim}.
Tie-break: lower total ERT across that dim, then alphabetical.
    """
wins = defaultdict(Counter)                    # dim -> Counter({alg: count})
ert_sums = defaultdict(lambda: defaultdict(float))  # dim -> {alg: total_ert}

for (dim, fid), (alg, ert) in best_by_df.items():
    if alg is None or not np.isfinite(ert):
        continue
    wins[dim][alg] += 1
    ert_sums[dim][alg] += float(ert)

result = {}
for dim, counter in wins.items():
    max_wins = max(counter.values())
    candidates = [a for a, c in counter.items() if c == max_wins]
    best = min(candidates, key=lambda a: (ert_sums[dim][a], a))  # tie-breaks
    result[dim] = best
result


{2: 'AVGNEWUOA_ros',
 3: 'AVGNEWUOA_ros',
 5: 'AVGNEWUOA_ros',
 10: 'IPOP-ACTCMA-ES_ros',
 20: 'IPOP-ACTCMA-ES_ros',
 40: 'IPOP-ACTCMA-ES_ros'}

In [6]:
import numpy as np
import pandas as pd

# Make sure 'dd' already exists
# (if not, run: dsl = cocopp.load('path/to/your/ppdata'); dd = dsl.dictByDimFunc())

targets = [1e-1, 1e-2, 1e-3, 1e-5, 1e-8]
rows = []  # reset before starting the full loop

for dim in sorted(dd.keys()):                      # e.g. [2, 3, 5, 10, 20, 40]
    for fid in sorted(dd[dim].keys()):
        for t in targets:
            algo_erts = []
            for ds in dd[dim][fid]:                # each algorithm
                ert = float(ds.detERT([t])[0])
                algo_erts.append((ds.algId, ert))
            
            finite = [(a, e) for (a, e) in algo_erts if np.isfinite(e)]

            if finite:
                best_alg, best_ert = min(finite, key=lambda x: x[1])
            else:
                best_alg, best_ert = None, np.inf

            rows.append({
                "dimension": dim,
                "function_id": fid,
                "target": t,
                "best_algorithm": best_alg,
                "best_ERT": best_ert
            })

# Build DataFrame
df_best = pd.DataFrame(rows)
df_best = df_best.sort_values(by=["dimension", "function_id", "target"]).reset_index(drop=True)

# Confirm dimensions included
print("✅ Unique dimensions in table:", df_best["dimension"].unique())
print(df_best.head(15))


✅ Unique dimensions in table: [ 2  3  5 10 20 40]
    dimension  function_id        target          best_algorithm     best_ERT
0           2            1  1.000000e-08           AVGNEWUOA_ros     6.000000
1           2            1  1.000000e-05           AVGNEWUOA_ros     6.000000
2           2            1  1.000000e-03           AVGNEWUOA_ros     6.000000
3           2            1  1.000000e-02           AVGNEWUOA_ros     6.000000
4           2            1  1.000000e-01           AVGNEWUOA_ros     6.000000
5           2            2  1.000000e-08  1plus2mirser_brockhoff   434.200000
6           2            2  1.000000e-05  1plus2mirser_brockhoff   363.733333
7           2            2  1.000000e-03  1plus2mirser_brockhoff   321.866667
8           2            2  1.000000e-02           AVGNEWUOA_ros   264.866667
9           2            2  1.000000e-01           AVGNEWUOA_ros   209.266667
10          2            3  1.000000e-08           AVGNEWUOA_ros  1447.200000
11          2 

In [7]:
import os
os.makedirs("results", exist_ok=True)

df_best.to_csv("results/best_algos_2010.csv", index=False)
